# Simple: K-Anonymity Profiling with PAMOLA.CORE

**Goal**: Learn k-anonymity profiling basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Analyze privacy risks using quasi-identifiers
- Compare before/after k-anonymity metrics
- Understand re-identification vulnerabilities

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path
- Verifies all imports work correctly

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 01_k_anonymity_profiler_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.anonymity import KAnonymityProfilerOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Load data from `examples/data_examples/sample.csv`
- Auto-create sample data if file doesn't exist
- Inspect the dataset structure

**Expected output:** DataFrame with records including patient demographics and quasi-identifiers

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample patient data with quasi-identifiers
    sample_data = pd.DataFrame({
        'patient_id': range(1, 51),
        'age_group': ['20-30'] * 8 + ['30-40'] * 12 + ['40-50'] * 15 + ['50-60'] * 10 + ['60-70'] * 5,
        'gender': ['M', 'F'] * 25,
        'zip_code': ['10001', '10002', '10003', '10004', '10005'] * 10,
        'diagnosis': ['Type A', 'Type B', 'Type C', 'Type D', 'Type E'] * 10,
        'treatment_cost': np.random.randint(1000, 10000, 50)
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
- Create task directory for outputs
- Initialize TaskReporter for tracking
- Setup DataSource with our DataFrame
- Configure progress tracker
- **Configure quasi-identifiers** for analysis

**Configuration pattern (production-style):**
```python
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "quasi_identifiers": ["years_of_experience", "location_province"],  # Customize this!
        "id_fields": ["resume_id"]
    }
```

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "quasi_identifiers": ["years_of_experience", "location_province"],  # Fields for privacy analysis
        "id_fields": ["resume_id"]
    }

kwargs = create_config_kwargs()
quasi_identifiers = kwargs["quasi_identifiers"]
id_fields = kwargs["id_fields"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")
missing_fields = [field for field in quasi_identifiers if field not in df.columns]
if missing_fields:
    raise ValueError(
        f"❌ Fields {missing_fields} not found in dataset!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update 'quasi_identifiers' in create_config_kwargs()"
    )

print(f"✓ Quasi-identifiers: {', '.join(quasi_identifiers)}")
for field in quasi_identifiers:
    print(f"  {field:15s}: {df[field].nunique()} unique values")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="privacy_profiling",
    description="K-anonymity profiling of patient records using quasi-identifiers",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description=f"Analyzing privacy with {len(quasi_identifiers)} quasi-identifiers",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Create KAnonymityProfilerOperation with full config
- Use `operation.execute()` with explicit execution configs
- Monitor progress through tracker

**Key parameters:**
- `quasi_identifiers`: Fields that could identify individuals
- `analysis_mode='ANALYZE'`: Generate metrics and reports
- `threshold_k=5`: Records with k<5 are vulnerable
- `export_metrics=True`: Save detailed metrics
- `id_fields=id_fields`: ID columns used for grouping or record tracking
- `generate_visualization=True`: Create charts
- `save_output=True`: Save to files
- `force_recalculation=False`: Use cache if available

In [ ]:
# Create operation with production-style configuration
operation = KAnonymityProfilerOperation(
    # Core parameters
    quasi_identifiers=quasi_identifiers,
    analysis_mode='ANALYZE',
    
    # Privacy thresholds
    threshold_k=5,                   # k<5 = vulnerable
    max_combinations=50,             # Max QI combinations to analyze
    
    # ID fields for tracking
    id_fields=id_fields,
    
    # Output configuration
    export_metrics=True,
    output_field_suffix='k_anon',
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Quasi-identifiers: {', '.join(operation.quasi_identifiers)}")
print(f"  Analysis mode:     {operation.analysis_mode.value}")
print(f"  Threshold k:       {operation.threshold_k}")
print(f"  Export metrics:    {operation.export_metrics}")
print(f"  Visualizations:    {operation.generate_visualization}")
print(f"  Save output:       {operation.save_output}")
print(f"  Force recalc:      {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing k-anonymity profiling...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Load the profiling results from task_dir
- Review k-anonymity metrics
- Identify vulnerable records

**Generated artifacts:**
- K-anonymity summary JSON in output/
- Vulnerable records list in output/
- Visualizations in visualizations/

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load summary file
output_files = list(task_dir.glob('output/*summary*.json'))
if output_files:
    summary_file = output_files[0]
    
    with open(summary_file, 'r') as f:
        summary_data = json.load(f)
    
    print("\n" + "=" * 80)
    print("📊 K-ANONYMITY ANALYSIS RESULTS")
    print("=" * 80)
    
    # Display metrics for quasi-identifier combination
    for qi_name, metrics in summary_data.items():
        print(f"\n🔍 Combination: {metrics['fields']}")
        print(f"   Min k:             {metrics['min_k']}")
        print(f"   Mean k:            {metrics['mean_k']:.2f}")
        print(f"   Vulnerable %:      {metrics['vulnerable_percent']:.2f}%")
        print(f"   Entropy:           {metrics['entropy']:.4f}")
    
    print("\n" + "=" * 80)
    print("✨ PRIVACY RISK SUMMARY")
    print("=" * 80)
    
    # Overall assessment
    first_combo = list(summary_data.values())[0]
    vuln_pct = first_combo['vulnerable_percent']
    
    if vuln_pct < 5:
        risk_level = "LOW"
        emoji = "✅"
    elif vuln_pct < 20:
        risk_level = "MEDIUM"
        emoji = "⚠️"
    else:
        risk_level = "HIGH"
        emoji = "❌"
    
    print(f"  {emoji} Risk Level:        {risk_level}")
    print(f"  📊 Total Records:      {len(df)}")
    print(f"  🔒 Min k-value:        {first_combo['min_k']}")
    print(f"  📈 Mean k-value:       {first_combo['mean_k']:.2f}")
    print(f"  ⚠️  Vulnerable %:       {vuln_pct:.2f}%")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Additional Metrics:")
        k_anon_metrics = result.metrics.get('k_anonymity', {})
        for key, value in list(k_anon_metrics.items())[:5]:
            if isinstance(value, dict):
                print(f"  {key}:")
                for k, v in list(value.items())[:3]:
                    if isinstance(v, float):
                        print(f"    {k:20s}: {v:.4f}")
                    else:
                        print(f"    {k:20s}: {v}")
    
    print(f"\n💡 Lower k-values = Higher re-identification risk!")
else:
    print("⚠️  No summary file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Check all generated files
- Navigate to directories for manual inspection

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # JSON summaries
├── metrics/          # Detailed metrics
├── visualizations/   # PNG charts
└── config/           # Operation config
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: K-anonymity metrics and vulnerability analysis
2. **Summary Data**: Privacy risk assessment across QI combinations
3. **Visualizations**: Charts showing vulnerability distribution
4. **Vulnerable Records**: List of high-risk record IDs

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # Display metadata
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")
            if 'operation' in metadata:
                op = metadata['operation']
                print(f"   Operation: {op.get('class', 'N/A')}.{op.get('function', 'N/A')}")
            
            # Display k-anonymity metrics
            if 'k_anonymity' in metrics:
                print("\n📊 K-ANONYMITY METRICS:")
                k_anon = metrics['k_anonymity']
                for qi_name, qi_metrics in k_anon.items():
                    print(f"\n   QI Combination: {qi_name}")
                    print(f"      Min k:           {qi_metrics.get('min_k', 'N/A')}")
                    print(f"      Mean k:          {qi_metrics.get('mean_k', 'N/A'):.2f}")
                    print(f"      Vulnerable %:    {qi_metrics.get('vulnerable_percent', 'N/A'):.2f}%")
            
            # Display performance metrics
            if 'performance' in metrics:
                print("\n⚡ PERFORMANCE:")
                perf = metrics['performance']
                print(f"   Duration: {perf.get('duration_seconds', 0):.4f}s")
                print(f"   Records Processed: {perf.get('records_processed', 0):,}")
        
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. K-ANONYMITY SUMMARY (NEWEST)
print("\n\n2️⃣ K-ANONYMITY SUMMARY:")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    summary_files = list(output_dir.glob('*summary*.json'))

    if not summary_files:
        print("⚠️  No summary files found")
    else:
        # Sort by modification time (newest first)
        summary_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_summary_file = summary_files[0]

        mtime = datetime.fromtimestamp(latest_summary_file.stat().st_mtime)
        print(f"✓ Found {len(summary_files)} summary file(s)")
        print(f"📄 Loading LATEST: {latest_summary_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_summary_file.stat().st_size / 1024:.1f} KB\n")

        try:
            with open(latest_summary_file, 'r', encoding='utf-8') as f:
                summary_data = json.load(f)

            print("📊 K-ANONYMITY BY QI COMBINATION:")
            print("=" * 90)
            print(f"{'QI Key':<20} {'Fields':<30} {'Min k':>8} {'Mean k':>10} {'Vuln %':>10} {'Entropy':>10}")
            print("-" * 90)

            for qi_key, data in summary_data.items():
                fields = data.get("fields", qi_key)
                fields_str = fields[:27] + "..." if len(fields) > 30 else fields

                min_k = data.get("min_k", 0)
                mean_k = data.get("mean_k", 0)
                vuln = data.get("vulnerable_percent", 0)
                entropy = data.get("entropy", 0)

                print(
                    f"{qi_key:<20} "
                    f"{fields_str:<30} "
                    f"{min_k:>8} "
                    f"{mean_k:>10.2f} "
                    f"{vuln:>9.1f}% "
                    f"{entropy:>10.4f}"
                )

            # Decision summary
            print("\n🧠 PRIVACY DECISION SUMMARY")
            print("-" * 80)

            for qi_key, data in summary_data.items():
                min_k = data.get("min_k", 0)
                vuln = data.get("vulnerable_percent", 100)

                print(f"QI: {qi_key}")
                if min_k == 1 and vuln <= 5:
                    print("   ⚠️  Minor k-anonymity violation (localized tail)")
                    print("   → Recommend suppression or local generalization")
                elif min_k >= 2:
                    print("   ✅ Satisfies practical k-anonymity")
                else:
                    print("   ❌ High k-anonymity risk detected")

        except Exception as e:
            print(f"❌ Error reading summary: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            latest_viz_batch.sort(key=lambda x: x.name)
            
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            for i, viz_file in enumerate(latest_viz_batch, 1):
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

# 4. VULNERABLE RECORDS (NEWEST)
print("\n\n4️⃣ VULNERABLE RECORDS:")
print("-" * 80)

vuln_files = list(output_dir.glob('vulnerable_records*.json'))

if not vuln_files:
    print("   ℹ️  No vulnerable records file found")
else:
    # Sort by modification time (newest first)
    vuln_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    latest_vuln_file = vuln_files[0]

    mtime = datetime.fromtimestamp(latest_vuln_file.stat().st_mtime)
    print(f"✓ Found {len(vuln_files)} vulnerable records file(s)")
    print(f"📄 Loading LATEST: {latest_vuln_file.name}")
    print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Size: {latest_vuln_file.stat().st_size / 1024:.1f} KB\n")

    try:
        with open(latest_vuln_file, 'r', encoding='utf-8') as f:
            vuln_data = json.load(f)

        print("📋 VULNERABLE RECORDS BY QI COMBINATION")
        print("=" * 90)
        print(
            f"{'QI Key':<20} "
            f"{'Vulnerable':>12} "
            f"{'Vuln %':>10} "
            f"{'Groups':>10} "
            f"{'Sample IDs'}"
        )
        print("-" * 90)

        for qi_key, info in vuln_data.items():
            total_vuln = info.get("total_vulnerable", 0)
            vuln_pct = info.get("vulnerable_percent", 0)
            groups = info.get("vulnerable_groups", 0)
            sample_ids = info.get("sample_ids", [])

            sample_str = ", ".join(map(str, sample_ids[:8])) if sample_ids else "-"

            print(
                f"{qi_key:<20} "
                f"{total_vuln:>12,} "
                f"{vuln_pct:>9.1f}% "
                f"{groups:>10} "
                f"{sample_str}"
            )

        # Risk interpretation
        print("\n🧠 VULNERABILITY INTERPRETATION")
        print("-" * 80)

        for qi_key, info in vuln_data.items():
            vuln_pct = info.get("vulnerable_percent", 100)
            total_vuln = info.get("total_vulnerable", 0)

            print(f"QI: {qi_key}")
            if vuln_pct <= 5:
                print("   ✅ Low exposure")
                print(f"   → Only {total_vuln:,} records affected (localized tail)")
                print("   → Recommend suppression or local generalization")
            elif vuln_pct <= 15:
                print("   ⚠️  Moderate exposure")
                print("   → Review aggregation thresholds")
            else:
                print("   ❌ High exposure")
                print("   → Aggregation required")

    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Error reading vulnerable records: {e}")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure k-anonymity profiling with quasi-identifiers  
✅ Execute using operation.execute()  
✅ Analyze privacy risk and vulnerability metrics

**Key takeaways:**
- K-anonymity measures re-identification risk using quasi-identifiers
- Lower k-values indicate higher privacy risk
- Vulnerable records (k < threshold) need additional protection
- Multiple QI combinations reveal different privacy vulnerabilities
- Entropy measures the diversity of equivalence classes
- Analysis mode provides detailed metrics without modifying data

---

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_k_anonymity_profiler_advanced.ipynb`** includes:
- ENRICH mode to add k-values as DataFrame columns
- Multiple quasi-identifier combinations analysis
- Custom threshold configurations
- L-diversity and t-closeness extensions
- Large dataset optimization with chunked processing
- Integration with anonymization operations

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)
- 🎓 [Privacy Metrics Guide](../../docs/privacy_metrics.md)